Activity:

Purpose: You'll research these questions using your favorite LLM / other means. Then we'll discuss as a group.
 

Parquet
1. Is it possible to change the statistics that parquet stores as metadata (e.g., can we have it store standard deviations of the columns)?
- **No.** The Parquet format spec fixes the statistics schema per column chunk to `min`/`max` (legacy) or `min_value`/`max_value` (current), `null_count`, and `distinct_count`. No writer — PyArrow, Spark, Polars — exposes a way to inject a custom statistic like standard deviation into that interpreted metadata. Query engines only ever read those four fields for pruning.
- If you want something like stddev associated with a file, your real options are: store it as generic (uninterpreted) Parquet key-value metadata, or keep it in an external catalog (Hive Metastore, Iceberg/Delta table metadata) — either way it's on you to compute and consume it; no engine will use it for optimization.

2. Parquet forms row groups from the set of rows. Can you manually edit the starting and endng records for the row groups? If so, how?
- Not by editing an existing file in place — you'd rewrite it. **At write time**, though, yes:
  - **PyArrow**: use `ParquetWriter.write_table()`, calling it once per row group with whatever slice of rows you want in that group. Each call becomes its own row group.
  - **Spark**: `parquet.block.size` / `spark.sql.parquet.block.size` controls row-group size in **bytes**, not exact record boundaries — less precise than the PyArrow approach.

3. Can you inspect statistics like min and max for row groups? If so, how?
Yes, via PyArrow:
```python
import pyarrow.parquet as pq
pf = pq.ParquetFile("file.parquet")
rg = pf.metadata.row_group(0)
col = rg.column(0)
col.statistics.min, col.statistics.max, col.statistics.null_count
```

4. Review and run notebook: rle_and_parquet_demo
   - Perform each task

Catalyst
5. Revisit the Spark SQL and Dataframes notebook.  
  Create the Spark Session and run sections 13-Aggregate on Columns and 14-Joins at Scale.
  For the Broadcast Hash Join object df_join_bhj:  
  How can you see the plans below? Write and run the code to show each of the following:  
  == Parsed Logical Plan ==  
  == Analyzed Logical Plan ==   
  == Optimized Logical Plan ==  
  == Physical Plan ==  

Briefly review the plans and note the differences

### 4. `rle_and_parquet_demo` notebook

Completed — Task 1 fills in a loop that runs `rle_encode()` on all three datasets and checks round-trip correctness via `rle_decode()`. The file is attached below.

**Task 2 (compression ratio) / Task 3 (row groups & encodings) — does it make sense?**

- **Compression tracks adjacency of repeats, not uniqueness.** `repetitive` (3 unique values, long contiguous runs) → 1000:1. `moderate` (same 3 values, alternating every 2) → 2:1. `no_runs_values` (4 unique values, no two adjacent equal) → 1:1, despite having *fewer* unique values than `moderate`. RLE only cares about local repetition, not cardinality.
- **Row group structure is identical across all three files** (3 groups of 1000 rows), since that's set explicitly by `row_group_size=1000` — it doesn't depend on the data pattern at all.
- **Min/max pruning value is the real payoff of this exercise.** For `repetitive`, each row group has min == max (a single distinct value per group), so a query filtering on `status` could skip 2 of 3 row groups entirely. For `moderate` and `no_runs`, every row group spans the full value range (min=Canada/France, max=US in both), so min/max gives **zero pruning benefit** — even though `moderate` still compresses fine via RLE_DICTIONARY. This is the key separation: encoding efficiency (file size) and statistics-based pruning (query skip-ability) are independent properties, and this dataset design makes that visible.
- **Encodings** are `('PLAIN', 'RLE', 'RLE_DICTIONARY')` across all three files regardless of pattern — PyArrow's default dictionary-encodes low-cardinality strings, then RLE-encodes the resulting dictionary IDs. The *choice* of encoding doesn't change; its *effectiveness* does, which is what the compression ratios capture.---

## Catalyst — Question 5

Sections 13 (Aggregate on Columns) and 14 (Joins at Scale) live in `spark_sql_and_dataframes.ipynb`; `df_join_bhj` is already defined there. To get all four plan sections in one shot:

```python
df_join_bhj.explain(mode="extended")
```

This is the mode that prints `== Parsed Logical Plan ==`, `== Analyzed Logical Plan ==`, `== Optimized Logical Plan ==`, and `== Physical Plan ==` — confirmed directly against the official PySpark `explain()` docs, which show exactly that four-section output for `extended`.

**For reference, corrected, the full set of `explain()` modes:**
- `mode="simple"` → physical plan only (the default)
- `mode="extended"` → all four plans (parsed, analyzed, optimized, physical) — **use this one**
- `mode="formatted"` → physical plan split into an outline section + node-detail section
- `mode="cost"` → **optimized logical plan** + statistics (not the physical plan — I had this backwards before)

**What to look for across the four plans:**
- **Parsed Logical Plan**: raw tree from parsing, unresolved references (`UnresolvedAttribute`/`UnresolvedRelation`), no type checking yet.
- **Analyzed Logical Plan**: Catalyst has resolved everything against the catalog — types are concrete, the broadcast hint is attached — but no rewriting for performance yet.
- **Optimized Logical Plan**: rule-based rewrites applied (predicate pushdown, column pruning, constant folding); still engine-agnostic.
- **Physical Plan**: concrete execution strategy — this is where `BroadcastHashJoin`, `BroadcastExchange`, and shuffle (`Exchange`) nodes actually show up, matching the notebook's own markdown notes about the shuffle happening on the broadcast side before the hash table is built and sent to executors.

The first two plans are about **correctness/resolution**; the last two are about **execution strategy** — that's the throughline worth noting when you compare them side by side.

### Purpose: Demonstrate Run Length Encoding (RLE) and how it's used by Parquet

In [2]:
import pyarrow as pa
import pyarrow.parquet as pq
import pandas as pd
import os

#### 1. RLE Implementation

In [3]:
def rle_encode(values):
    """Return [(value, run_length), ...]."""
    if not values:
        return []

    encoded = []
    current = values[0]
    count = 1

    for value in values[1:]:
        if value == current: # if next value is duplicate, increment count
            count += 1
        else:
            encoded.append((current, count)) # otherwise end the tuple
            current = value
            count = 1

    encoded.append((current, count))
    return encoded


def rle_decode(encoded):
    """Reconstruct the original sequence."""
    values = []

    for value, count in encoded:
        values.extend([value] * count)

    return values

#### 2. Create three different data patterns

In [4]:
repetitive = (
    ["US"] * 1000 +
    ["Spain"] * 1000 +
    ["France"] * 1000
)

moderate = (
    ["US", "US", "Spain", "Spain", "France", "France"] * 500
)

# Deterministic pattern without runs
no_runs_values = [
    ["US", "Canada", "UK", "France"][i % 4]
    for i in range(3000)
]

============================================================  

**Task 1: Test the `rle_encode()` function on each dataset**

============================================================  

In [8]:
rle_encode(repetitive)

[('US', 1000), ('Spain', 1000), ('France', 1000)]

#### 3. Look at the RLE representations

In [5]:
for name, values in [
    ("Highly repetitive", repetitive),
    ("Moderately repetitive", moderate),
    ("Random-looking", no_runs_values)
]:

    encoded = rle_encode(values)

    print("\n", name)
    print("-" * 50)
    print("Original values:", len(values))
    print("RLE runs:", len(encoded))
    print("Compression ratio:", len(values) / len(encoded))

    print("First 10 runs:")
    print(encoded[:10])


 Highly repetitive
--------------------------------------------------
Original values: 3000
RLE runs: 3
Compression ratio: 1000.0
First 10 runs:
[('US', 1000), ('Spain', 1000), ('France', 1000)]

 Moderately repetitive
--------------------------------------------------
Original values: 3000
RLE runs: 1500
Compression ratio: 2.0
First 10 runs:
[('US', 2), ('Spain', 2), ('France', 2), ('US', 2), ('Spain', 2), ('France', 2), ('US', 2), ('Spain', 2), ('France', 2), ('US', 2)]

 Random-looking
--------------------------------------------------
Original values: 3000
RLE runs: 3000
Compression ratio: 1.0
First 10 runs:
[('US', 1), ('Canada', 1), ('UK', 1), ('France', 1), ('US', 1), ('Canada', 1), ('UK', 1), ('France', 1), ('US', 1), ('Canada', 1)]


============================================================  

**Task 2: Notice the compression ratio from each dataset**

============================================================  

   - Notice details like compression ratio, row groups, and encodings


#### 4. Create Datasets and Write as Parquet

In [6]:
datasets = {
    "repetitive": repetitive,
    "moderate": moderate,
    "no_runs": no_runs_values
}

for name, values in datasets.items():

    df = pd.DataFrame({
        "status": values
    })

    # use pyarrow
    table = pa.Table.from_pandas(df)

    filename = f"{name}.parquet"

    # write as parquet file
    pq.write_table(
        table,
        filename,
        row_group_size=1000, # set row group size to partition into multiple groups
        use_dictionary=True
    )

    print(f"{filename}: {os.path.getsize(filename):,} bytes")

repetitive.parquet: 1,922 bytes
moderate.parquet: 2,027 bytes
no_runs.parquet: 2,030 bytes


#### 5. Inspect what Parquet Stored

In [7]:
for name in datasets:

    filename = f"{name}.parquet"

    pf = pq.ParquetFile(filename)

    print("\n" + "=" * 60)
    print(name.upper())
    print("=" * 60)

    print("Number of row groups:", pf.num_row_groups)

    for rg_number in range(pf.num_row_groups):

        rg = pf.metadata.row_group(rg_number)

        print("\nRow group:", rg_number)
        print("Rows:", rg.num_rows)

        for column_number in range(rg.num_columns):

            column = rg.column(column_number)

            print("Column:", column.path_in_schema)
            print("Encodings:", column.encodings)

            if column.statistics:
                print("Min:", column.statistics.min)
                print("Max:", column.statistics.max)
                print("Null count:", column.statistics.null_count)


REPETITIVE
Number of row groups: 3

Row group: 0
Rows: 1000
Column: status
Encodings: ('PLAIN', 'RLE', 'RLE_DICTIONARY')
Min: US
Max: US
Null count: 0

Row group: 1
Rows: 1000
Column: status
Encodings: ('PLAIN', 'RLE', 'RLE_DICTIONARY')
Min: Spain
Max: Spain
Null count: 0

Row group: 2
Rows: 1000
Column: status
Encodings: ('PLAIN', 'RLE', 'RLE_DICTIONARY')
Min: France
Max: France
Null count: 0

MODERATE
Number of row groups: 3

Row group: 0
Rows: 1000
Column: status
Encodings: ('PLAIN', 'RLE', 'RLE_DICTIONARY')
Min: France
Max: US
Null count: 0

Row group: 1
Rows: 1000
Column: status
Encodings: ('PLAIN', 'RLE', 'RLE_DICTIONARY')
Min: France
Max: US
Null count: 0

Row group: 2
Rows: 1000
Column: status
Encodings: ('PLAIN', 'RLE', 'RLE_DICTIONARY')
Min: France
Max: US
Null count: 0

NO_RUNS
Number of row groups: 3

Row group: 0
Rows: 1000
Column: status
Encodings: ('PLAIN', 'RLE', 'RLE_DICTIONARY')
Min: Canada
Max: US
Null count: 0

Row group: 1
Rows: 1000
Column: status
Encodings: ('PL

---

RLE_DICTIONARY creates a dictionary to map strings to IDs:

"United States" → 0  
"Spain"         → 1  
"France"        → 2

==============================================================================================  

**Task 3: Notice the row groups and encodings from each dataset. Does this info make sense?**

==============================================================================================  
